# Vamos criar um modelo predito de machine learning para prever o score de crédito do cliente
<p> Um resumo do que veremos aqui:<br>
    - Analise Exploratória e Gráficos<br>
    - Tratamento de dados missing <br>
    - Tratamento de outliers <br>
    - OneHotEncoding <br>
    - Engenharia de Atributos <br>
    - Tratamento de dados <br>
    - Normalização de dados <br>
    - Criação, teste e validação de um modelo de machine learning

## IMPORTAÇÃO DAS BIBLIOTECAS

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split 
from sklearn.preprocessing import StandardScaler 
from sklearn.preprocessing import MinMaxScaler 
from sklearn.preprocessing import LabelEncoder 
from sklearn.linear_model import LinearRegression 
from sklearn.metrics import r2_score 

## CARREGAMENTO DOS DADOS

In [ ]:
df = pd.read_excel("dados_credito.xlsx")

print("Dimensão inicial do dataset:", df.shape)
df.head()

## INSPEÇÃO INICIAL 

In [ ]:
df.info()
df.describe(include='all').T


## TRATAMENTO DE DADOS FALTANTES E INCONSISTÊNCIAS

In [ ]:
df.isnull().sum()

In [ ]:
df.groupby(['ULTIMO_SALARIO']).size()

In [ ]:
df.replace('SEM DADOS',np.nan, inplace = True)

In [ ]:
df['ULTIMO_SALARIO'] = df['ULTIMO_SALARIO'].fillna((df['ULTIMO_SALARIO'].median()))

In [ ]:
df['ULTIMO_SALARIO'] = df['ULTIMO_SALARIO'].astype(np.float64)
df['VL_IMOVEIS'] = df['VL_IMOVEIS'].astype(np.float64)
df['OUTRA_RENDA_VALOR'] = df['OUTRA_RENDA_VALOR'].astype(np.float64)
df['VALOR_TABELA_CARROS'] = df['VALOR_TABELA_CARROS'].astype(np.float64)

In [ ]:
df.groupby(['QT_FILHOS']).size()

In [ ]:
df['QT_FILHOS'] = df['QT_FILHOS'].fillna((df['QT_FILHOS'].mode()))

In [ ]:
df.groupby(['OUTRA_RENDA_VALOR']).size()

In [ ]:
df.groupby(['VALOR_TABELA_CARROS']).size()

In [ ]:
df.groupby(['QT_IMOVEIS']).size()

In [ ]:
df.duplicated().sum()

## ANÁLISE EXPLORATÓRIA (EDA)

In [ ]:
num = []
for i in df.columns[0:16].tolist():
        if df.dtypes[i] == 'int64' or df.dtypes[i] == 'float64':            
            print(i, ':' , df.dtypes[i]) 
            num.append(i)

In [ ]:
cat = []
for i in df.columns[0:48].tolist():
        if df.dtypes[i] == 'object' or df.dtypes[i] == 'category':            
            print(i, ':' , df.dtypes[i]) 
            cat.append(i)           

In [ ]:
plt.rcParams["figure.figsize"] = [15.00, 12.00]
plt.rcParams["figure.autolayout"] = True

f, axes = plt.subplots(2, 5) 

linha = 0
coluna = 0

for i in num:
    sns.boxplot(data = df, y=i, ax=axes[linha][coluna])
    coluna += 1
    if coluna == 5:
        linha += 1
        coluna = 0            

plt.show()

In [ ]:
plt.rcParams["figure.figsize"] = [15.00, 12.00]
plt.rcParams["figure.autolayout"] = True

f, axes = plt.subplots(4, 3) 

linha = 0
coluna = 0

for i in num:
    sns.histplot(data = df, x=i, ax=axes[linha][coluna])    
    coluna += 1
    if coluna == 3:
        linha += 1
        coluna = 0            

plt.show()

In [ ]:

plt.rcParams["figure.figsize"] = [15.00, 22.00]
plt.rcParams["figure.autolayout"] = True


f, axes = plt.subplots(3, 2) 

linha = 0
coluna = 0

for i in cat:    
    sns.countplot(data = df, x=i, ax=axes[linha][coluna])
    
    coluna += 1
    if coluna == 2:
        linha += 1
        coluna = 0            

plt.show()



In [ ]:
plt.rcParams["figure.figsize"] = (18, 8)

corr = df.select_dtypes(include=[np.number]).corr()

ax = sns.heatmap(corr, annot=True, cmap="coolwarm")
plt.title("Matriz de Correlação (somente variáveis numéricas)")
plt.show()

In [ ]:

sns.lmplot(x = "VL_IMOVEIS", y = "SCORE", data = df);

In [ ]:

sns.lmplot(x = "ULTIMO_SALARIO", y = "SCORE", data = df);

In [ ]:

sns.lmplot(x = "TEMPO_ULTIMO_EMPREGO_MESES", y = "SCORE", data = df);

## CRIAÇÃO DE NOVAS FEATURES

In [ ]:
idade_faixas = [0, 20, 30, 45, 55, 100]
idade_categoria = ["Até 20", "21 a 30", "31 a 45", "46 a 55", "Maior que 55"]

df["FAIXA_ETARIA"] = pd.cut(df["IDADE"], idade_faixas, labels=idade_categoria)
df["FAIXA_ETARIA"].value_counts()

In [ ]:
df['TEM_FILHOS'] = np.where(df['QT_FILHOS'] > 0, 'Sim', 'Não')
df["TEM_FILHOS"].value_counts()

In [ ]:
df['RENDA_TOTAL'] = df['ULTIMO_SALARIO'] + df['OUTRA_RENDA_VALOR']
df['RENDA_TOTAL'].head()

In [ ]:
df.groupby(['RENDA_TOTAL']).size()

In [ ]:
df['CATEGORIA_RENDA'] = pd.cut(
    df['RENDA_TOTAL'],
    bins=[0, 2500, 5000, 10000, 20000, float('inf')],
    labels=['Baixa', 'Média-Baixa', 'Média', 'Média-Alta', 'Alta']
)
df['CATEGORIA_RENDA'].value_counts()

In [ ]:
df.head()

## Pré Processamento dos Dados

In [ ]:
lb = LabelEncoder()

df['FAIXA_ETARIA'] = lb.fit_transform(df['FAIXA_ETARIA'])
df['OUTRA_RENDA'] = lb.fit_transform(df['OUTRA_RENDA'])
df['TRABALHANDO_ATUALMENTE'] = lb.fit_transform(df['TRABALHANDO_ATUALMENTE'])
df['ESTADO_CIVIL'] = lb.fit_transform(df['ESTADO_CIVIL'])
df['CASA_PROPRIA'] = lb.fit_transform(df['CASA_PROPRIA'])
df['ESCOLARIDADE'] = lb.fit_transform(df['ESCOLARIDADE'])
df['UF'] = lb.fit_transform(df['UF'])

df.dropna(inplace = True)

In [ ]:
df.head(5)

In [ ]:
df.info()

In [ ]:
target = df.iloc[:,15:16]

In [ ]:
preditoras = df.copy() 

del preditoras['SCORE'] 

preditoras.head()

In [ ]:
X_treino, X_teste, y_treino, y_teste = train_test_split(preditoras, target, test_size = 0.3, random_state = 459)

In [ ]:
sc = MinMaxScaler()
X_treino_normalizados = sc.fit_transform(X_treino)
X_teste_normalizados = sc.transform(X_teste)

## Criar, avaliar e testar nosso modelo preditivo 

In [ ]:
scaler = StandardScaler()
X_treino_normalizados = scaler.fit_transform(X_treino)
X_teste_normalizados = scaler.transform(X_teste)

modelo = LinearRegression(fit_intercept=True)
modelo.fit(X_treino_normalizados, y_treino)

In [ ]:
r2_score(y_teste, modelo.fit(X_treino_normalizados, y_treino).predict(X_teste_normalizados))

In [ ]:
UF = 2
IDADE = 42 
ESCOLARIDADE = 1
ESTADO_CIVIL = 2
QT_FILHOS = 1
CASA_PROPRIA = 1
QT_IMOVEIS = 1
VL_IMOVEIS = 300000
OUTRA_RENDA = 1
OUTRA_RENDA_VALOR = 2000 
TEMPO_ULTIMO_EMPREGO_MESES = 18 
TRABALHANDO_ATUALMENTE = 1
ULTIMO_SALARIO = 5400.0
QT_CARROS = 4
VALOR_TABELA_CARROS = 70000
FAIXA_ETARIA = 3

novos_dados = [UF, IDADE, ESCOLARIDADE, ESTADO_CIVIL, QT_FILHOS,CASA_PROPRIA,QT_IMOVEIS,VL_IMOVEIS,OUTRA_RENDA,
               OUTRA_RENDA_VALOR,TEMPO_ULTIMO_EMPREGO_MESES,TRABALHANDO_ATUALMENTE,ULTIMO_SALARIO,QT_CARROS,
               VALOR_TABELA_CARROS, FAIXA_ETARIA]


X = np.array(novos_dados).reshape(1, -1)
X = sc.transform(X)

print("Score de crédito previsto para esse cliente:", modelo.predict(X))